In [ ]:
# Install once. pyspark bundles Spark itself (needs Java, already present here).
# kafka-python is a pure-Python client for talking to a Kafka broker.
# (If they're already installed, pip just confirms and moves on.)
import sys, subprocess                                  # to run pip from inside the notebook
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--break-system-packages", "pyspark", "kafka-python-ng"])  # install both
print("done")                                           # marker so we know the cell finished

done


In [ ]:
import os                                              # to set an env var for Spark
os.environ["PYSPARK_PYTHON"] = sys.executable           # make Spark use this same Python
from pyspark.sql import SparkSession, functions as F     # SparkSession + the column functions (F)

spark = (SparkSession.builder                            # start building a session
         .appName("intro")                               # a name shown in Spark's UI/logs
         .master("local[2]")                             # run a local 2-core "cluster"
         .config("spark.ui.showConsoleProgress", "false")# quieter output in a notebook
         .config("spark.sql.shuffle.partitions", "4")    # small shuffle width for tiny data
         .getOrCreate())                                  # create it (or reuse if one exists)
spark.sparkContext.setLogLevel("ERROR")                  # hide Spark's chatty INFO/WARN logs
print("Spark version:", spark.version)                   # confirm Spark is up and its version

Spark version: 4.0.2


In [ ]:

# Build a tiny DataFrame from Python rows (normally you'd read files/Kafka/a database)
orders = spark.createDataFrame(
    [("North", "paid",   100),                          # (region, status, amount) rows
     ("South", "paid",   200),
     ("North", "unpaid", 50),
     ("South", "paid",   300),
     ("East",  "paid",   150)],
    ["region", "status", "amount"])                      # the column names

orders.show()                                           # an ACTION: prints the table
orders.printSchema()                                    # show inferred column names + types
print("Row count:", orders.count())                     # count() is also an action

+------+------+------+
|region|status|amount|
+------+------+------+
| North|  paid|   100|
| South|  paid|   200|
| North|unpaid|    50|
| South|  paid|   300|
|  East|  paid|   150|
+------+------+------+

root
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: long (nullable = true)

Row count: 5


In [ ]:
# These are TRANSFORMATIONS — they return a new DataFrame but compute nothing yet
paid = orders.filter(F.col("status") == "paid")         # keep only paid orders (lazy)
by_region = paid.groupBy("region").agg(                  # group + aggregate (lazy)
    F.sum("amount").alias("revenue"))                    # name the summed column "revenue"

print("Nothing has executed yet. Here is Spark's plan:\n")
by_region.explain()                                     # show the physical plan (the DAG)

print("\nNow we call an ACTION (show) — Spark runs the whole plan:")
by_region.show()                                        # triggers execution

Nothing has executed yet. Here is Spark's plan:

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#0], functions=[sum(amount#2L)])
   +- Exchange hashpartitioning(region#0, 4), ENSURE_REQUIREMENTS, [plan_id=65]
      +- HashAggregate(keys=[region#0], functions=[partial_sum(amount#2L)])
         +- Project [region#0, amount#2L]
            +- Filter (isnotnull(status#1) AND (status#1 = paid))
               +- Scan ExistingRDD[region#0,status#1,amount#2L]



Now we call an ACTION (show) — Spark runs the whole plan:
+------+-------+
|region|revenue|
+------+-------+
| North|    100|
| South|    500|
|  East|    150|
+------+-------+



In [ ]:
output_path = "/tmp/by_region_output.csv"
by_region.coalesce(1).write.mode("overwrite").csv(output_path)
print(f"DataFrame saved to: {output_path}/")

NameError: name 'by_region' is not defined

In [ ]:
orders.createOrReplaceTempView("orders")            # expose the DataFrame as SQL table "orders"

result = spark.sql("""                                  -- ordinary SQL, run by Spark
    SELECT region, SUM(amount) AS revenue                -- total amount per region
    FROM orders
    WHERE status = 'paid'                                -- only paid orders
    GROUP BY region
    ORDER BY revenue DESC                                -- biggest region first
""")
result.show()                                           # action: run the SQL and print

In [ ]:
# Write a tiny CSV to disk so we have something to read
csv_path = "/tmp/sales.csv"                             # where to put the file
with open(csv_path, "w") as f:                          # open the file for writing
    f.write("region,amount\n")                          # header row
    f.write("North,120\nSouth,80\nNorth,200\nEast,60\n")  # four data rows

df_csv = (spark.read                                    # Spark's reader
          .option("header", True)                       # first line is the header
          .option("inferSchema", True)                  # detect column types automatically
          .csv(csv_path))                                # read the CSV path (could be a folder)

df_csv.show()                                           # show what we loaded
df_csv.groupBy("region").agg(F.avg("amount").alias("avg_amount")).show()  # quick

+------+------+
|region|amount|
+------+------+
| North|   120|
| South|    80|
| North|   200|
|  East|    60|
+------+------+

+------+----------+
|region|avg_amount|
+------+----------+
| North|     160.0|
|  East|      60.0|
| South|      80.0|
+------+----------+



In [ ]:
spark.stop()                                         # shut down the Spark session
print("Spark stopped.")                                 # confirmation

Spark stopped.


In [ ]:
import kafka                                          # the kafka-python client library
from kafka import KafkaProducer, KafkaConsumer            # the two main classes you use
print("kafka-python version:", kafka.__version__)         # confirm it imported
print("Main classes available:", KafkaProducer.__name__, "and", KafkaConsumer.__name__)

kafka-python version: 2.2.3
Main classes available: KafkaProducer and KafkaConsumer


In [ ]:
RUN_KAFKA = False                                    # set True only when a broker is running

if RUN_KAFKA:                                            # guard so this notebook runs without a broker
    import json                                          # to turn dicts into bytes
    producer = KafkaProducer(                            # connect a producer to the broker
        bootstrap_servers="localhost:9092",             # broker address
        value_serializer=lambda v: json.dumps(v).encode())  # serialise each value to JSON bytes
    for i in range(5):                                   # publish 5 example events
        event = {"order_id": i, "amount": 100 + i}       # the event payload
        producer.send("orders", value=event)            # append the event to topic "orders"
    producer.flush()                                     # make sure everything is sent
    producer.close()                                    # close the connection
    print("Sent 5 events to topic 'orders'.")
else:
    print("RUN_KAFKA is False — skipping (no broker). The code above is real; flip the flag to run it.")

RUN_KAFKA is False — skipping (no broker). The code above is real; flip the flag to run it.


In [ ]:
if RUN_KAFKA:                                         # again, only with a live broker
    import json                                          # to decode JSON bytes back to dicts
    consumer = KafkaConsumer(                            # connect a consumer
        "orders",                                       # the topic to read
        bootstrap_servers="localhost:9092",             # broker address
        group_id="my-group",                            # belong to this consumer group
        auto_offset_reset="earliest",                   # if no saved offset, start from the beginning
        consumer_timeout_ms=3000,                        # stop waiting after 3s of no messages
        value_deserializer=lambda b: json.loads(b.decode()))  # decode JSON bytes to a dict
    for msg in consumer:                                 # iterate over incoming events
        print(f"partition={msg.partition} offset={msg.offset} value={msg.value}")
    consumer.close()                                    # close the connection
else:
    print("RUN_KAFKA is False — skipping (no broker). This is the real consumer API.")

RUN_KAFKA is False — skipping (no broker). This is the real consumer API.
